In [21]:
# import required modules
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)



In [22]:
# dataframe for EV dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

In [23]:
df.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,5YJSA1E65N,Yakima,Granger,WA,98932.0,2022,TESLA,MODEL S,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,0.0,15.0,187279214,POINT (-120.1871 46.33949),PACIFICORP,5.307700e+10
1,KNDC3DLC5N,Yakima,Yakima,WA,98902.0,2022,KIA,EV6,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,0.0,15.0,210098241,POINT (-120.52041 46.59751),PACIFICORP,5.307700e+10
2,5YJYGDEEXL,Snohomish,Everett,WA,98208.0,2020,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,291.0,0.0,44.0,121781950,POINT (-122.18637 47.89251),PUGET SOUND ENERGY INC,5.306104e+10
3,3C3CFFGE1G,Yakima,Yakima,WA,98908.0,2016,FIAT,500,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,84.0,0.0,14.0,180778377,POINT (-120.60199 46.59817),PACIFICORP,5.307700e+10
4,KNDCC3LD5K,Kitsap,Bremerton,WA,98312.0,2019,KIA,NIRO,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,0.0,26.0,2581225,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,5.303508e+10


Data Cleaning
  .Check for missing or null values
  .Check for rows with duplicate values
  .Drop non-required rows
  . Check data type
  .Understand dataset

In [24]:
# Initial data exploration
print("Dataset shape:", df.shape)
print("="*55)
print("\nMissing values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


Dataset shape: (250659, 17)

Missing values:
VIN (1-10)                                             0
County                                                 6
City                                                   6
State                                                  0
Postal Code                                            6
Model Year                                             0
Make                                                   0
Model                                                  0
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                        21
Base MSRP                                             21
Legislative District                                 583
DOL Vehicle ID                                         0
Vehicle Location                                      14
Electric Utility                                       6
2020 Census Tract                          

In [25]:
# Data Cleaning - Enhanced
print("Original shape:", df.shape)

# Check for missing values more thoroughly
null_features = [features for features in df.columns if df[features].isnull().sum() >= 1]
print("\nFeatures with missing values:")
for f in null_features:
    null_percent = np.round(df[f].isnull().mean() * 100, 5)
    print(f"{f}: {df[f].isnull().sum()} missing values ({null_percent}%)")

# Drop columns with high missing rates
high_missing = [f for f in null_features if df[f].isnull().mean() > 0.3]
if high_missing:
    print(f"\nDropping columns with >30% missing: {high_missing}")
    df.drop(high_missing, axis=1, inplace=True)

# Drop rows with missing values in important columns
important_cols = ['Electric Range', 'Model Year', 'Make', 'Model', 'Electric Vehicle Type']
df = df.dropna(subset=important_cols)

print(f"\nShape after dropping: {df.shape}")

# Drop non-required columns
columns_to_remove = ['VIN (1-10)', 'DOL Vehicle ID', 'Vehicle Location', 
                    '2020 Census Tract', 'County', 'City', 'Postal Code',
                    'State', 'Electric Utility']  # Remove Electric Utility due to high cardinality
df.drop(columns=columns_to_remove, inplace=True, axis=1, errors='ignore')

print(f"Shape after dropping columns: {df.shape}")

Original shape: (250659, 17)

Features with missing values:
County: 6 missing values (0.00239%)
City: 6 missing values (0.00239%)
Postal Code: 6 missing values (0.00239%)
Electric Range: 21 missing values (0.00838%)
Base MSRP: 21 missing values (0.00838%)
Legislative District: 583 missing values (0.23259%)
Vehicle Location: 14 missing values (0.00559%)
Electric Utility: 6 missing values (0.00239%)
2020 Census Tract: 6 missing values (0.00239%)

Shape after dropping: (250638, 17)
Shape after dropping columns: (250638, 8)


FEATURE ENGINEERING

In [26]:
# Feature Engineering - Enhanced
from datetime import date

date_now = date.today()
year_now = date_now.year

# Create car age
df['Car Age'] = year_now - df['Model Year']

# Drop Model Year
df.drop('Model Year', inplace=True, axis=1)

# Check target distribution
print("="*55)
print("Target distribution:")
print(df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts())
print("="*55)
print(f"\nTarget proportions:")
print(df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts(normalize=True))

# Manual encoding for target
df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] = np.where(
    df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] == 'Not eligible due to low battery range', 0,
    np.where(df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'] == 'Clean Alternative Fuel Vehicle Eligible', 1, 2)
)

print("="*55)
print(f"\nEncoded target distribution:")
print(df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts())

Target distribution:
Clean Alternative Fuel Vehicle (CAFV) Eligibility
Eligibility unknown as battery range has not been researched    152155
Clean Alternative Fuel Vehicle Eligible                          74966
Not eligible due to low battery range                            23517
Name: count, dtype: int64

Target proportions:
Clean Alternative Fuel Vehicle (CAFV) Eligibility
Eligibility unknown as battery range has not been researched    0.607071
Clean Alternative Fuel Vehicle Eligible                         0.299101
Not eligible due to low battery range                           0.093829
Name: proportion, dtype: float64

Encoded target distribution:
Clean Alternative Fuel Vehicle (CAFV) Eligibility
2    152155
1     74966
0     23517
Name: count, dtype: int64


In [27]:
# Handle high cardinality categorical features
def reduce_cardinality(series, threshold=20):
    """Reduce cardinality by grouping infrequent categories"""
    counts = series.value_counts()
    mask = series.isin(counts.index[counts >= threshold])
    series = series.where(mask, 'Other')
    return series

# Apply to high cardinality features
if 'Make' in df.columns:
    df['Make'] = reduce_cardinality(df['Make'], threshold=100)
if 'Model' in df.columns:
    df['Model'] = reduce_cardinality(df['Model'], threshold=50)

print("Cardinality after reduction:")
for col in ['Make', 'Model', 'Electric Vehicle Type']:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")

Cardinality after reduction:
Make: 37 unique values
Model: 129 unique values
Electric Vehicle Type: 2 unique values


In [28]:
# Split data 
from sklearn.model_selection import train_test_split

# Define features and target
X = df.drop('Clean Alternative Fuel Vehicle (CAFV) Eligibility', axis=1)
y = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility']

# Initial split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:")
print(y_train.value_counts(normalize=True))
print(f"\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (200510, 7)
Test set: (50128, 7)

Training target distribution:
Clean Alternative Fuel Vehicle (CAFV) Eligibility
2    0.607072
1    0.299102
0    0.093826
Name: proportion, dtype: float64

Test target distribution:
Clean Alternative Fuel Vehicle (CAFV) Eligibility
2    0.607066
1    0.299094
0    0.093840
Name: proportion, dtype: float64


DATA PREPROCESSING

In [29]:
# Create preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Identify column types
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

# Define skewness transformation
def log_transform(X):
    return np.log1p(np.abs(X))

# Create transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log_transform', FunctionTransformer(log_transform)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Apply SMOTE ONLY on training data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("\nApplying preprocessing and SMOTE...")

# Create full pipeline with SMOTE
smote_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy='auto'))
])

# Fit and transform training data
X_train_processed, y_train_processed = smote_pipeline.fit_resample(X_train, y_train)

# Only transform test data (no SMOTE)
X_test_processed = preprocessor.transform(X_test)

print(f"\nTraining set after SMOTE: {X_train_processed.shape}")
print(f"Test set: {X_test_processed.shape}")
print(f"\nTraining target distribution after SMOTE:")
unique, counts = np.unique(y_train_processed, return_counts=True)
for val, count in zip(unique, counts):
    print(f"Class {val}: {count} samples ({count/len(y_train_processed)*100:.1f}%)")

Numeric features: ['Electric Range', 'Base MSRP', 'Legislative District', 'Car Age']
Categorical features: ['Make', 'Model', 'Electric Vehicle Type']

Applying preprocessing and SMOTE...

Training set after SMOTE: (365172, 172)
Test set: (50128, 172)

Training target distribution after SMOTE:
Class 0: 121724 samples (33.3%)
Class 1: 121724 samples (33.3%)
Class 2: 121724 samples (33.3%)


MODEL TRAINING & EVALUATION

In [30]:
# model evaluation function
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
import time

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Evaluate a single model"""
    start_time = time.time()
    
    # Train model
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Detailed report
    report = classification_report(y_test, y_test_pred, output_dict=False)
    
    return {
        'model': model_name,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'train_time': train_time,
        'classification_report': report
    }

# Define simpler models for initial testing
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Initialize models with reasonable parameters
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
    "SVM": SVC(random_state=42)  # Note: SVM can be slow with large datasets
}

# Evaluate each model
results = []
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Evaluating {name}")
    print('='*50)
    
    try:
        result = evaluate_model(model, X_train_processed, X_test_processed, 
                               y_train_processed, y_test, name)
        results.append(result)
        
        print(f"Training Accuracy: {result['train_accuracy']:.4f}")
        print(f"Testing Accuracy: {result['test_accuracy']:.4f}")
        print(f"Training F1: {result['train_f1']:.4f}")
        print(f"Testing F1: {result['test_f1']:.4f}")
        print(f"Training Time: {result['train_time']:.2f} seconds")
        print("\nClassification Report:")
        print(result['classification_report'])
        
    except Exception as e:
        print(f"Error with {name}: {str(e)}")

# Create results dataframe
if results:
    results_df = pd.DataFrame(results)
    print("\n" + "="*50)
    print("MODEL COMPARISON")
    print("="*50)
    print(results_df[['model', 'test_accuracy', 'test_f1', 'train_time']].sort_values('test_accuracy', ascending=False))





Evaluating Random Forest
Training Accuracy: 1.0000
Testing Accuracy: 1.0000
Training F1: 1.0000
Testing F1: 1.0000
Training Time: 15.95 seconds

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4704
           1       1.00      1.00      1.00     14993
           2       1.00      1.00      1.00     30431

    accuracy                           1.00     50128
   macro avg       1.00      1.00      1.00     50128
weighted avg       1.00      1.00      1.00     50128


Evaluating Logistic Regression
Training Accuracy: 0.9999
Testing Accuracy: 1.0000
Training F1: 0.9999
Testing F1: 1.0000
Training Time: 12.94 seconds

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4704
           1       1.00      1.00      1.00     14993
           2       1.00      1.00      1.00     30431

    accuracy                           1.00     50128
   

In [31]:
# Hyperparameter Tuning for the best model
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import joblib

# Let's tune RandomForest if it performed well, otherwise choose the best
if results:
    best_model_name = results_df.loc[results_df['test_accuracy'].idxmax(), 'model']
    print(f"\nPerforming hyperparameter tuning for {best_model_name}")
    
    if best_model_name == "Random Forest":
        # Define parameter grid for RandomForest
        param_grid = {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }
        
        # Use RandomizedSearchCV for faster search
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        search = RandomizedSearchCV(
            base_model, param_grid, 
            n_iter=20,  # Number of parameter settings sampled
            cv=3, 
            scoring='accuracy',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        
    elif best_model_name == "XGBoost":
        param_grid = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
        base_model = XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss')
        search = RandomizedSearchCV(
            base_model, param_grid, 
            n_iter=20,
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
    
    # Perform search
    print("Starting hyperparameter tuning...")
    search.fit(X_train_processed, y_train_processed)
    
    print(f"\nBest parameters: {search.best_params_}")
    print(f"Best cross-validation score: {search.best_score_:.4f}")
    
    # Evaluate best model
    best_model = search.best_estimator_
    y_test_pred = best_model.predict(X_test_processed)
    
    print("\nBest Model Performance:")
    print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Test F1 Score: {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # Save the best model
    joblib.dump(best_model, f'best_{best_model_name.lower().replace(" ", "_")}_model.pkl')
    joblib.dump(preprocessor, 'preprocessor.pkl')
    print(f"\nModel saved as 'best_{best_model_name.lower().replace(' ', '_')}_model.pkl'")


Performing hyperparameter tuning for Random Forest
Starting hyperparameter tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}
Best cross-validation score: 1.0000

Best Model Performance:
Test Accuracy: 1.0000
Test F1 Score: 1.0000

Detailed Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4704
           1       1.00      1.00      1.00     14993
           2       1.00      1.00      1.00     30431

    accuracy                           1.00     50128
   macro avg       1.00      1.00      1.00     50128
weighted avg       1.00      1.00      1.00     50128


Model saved as 'best_random_forest_model.pkl'


In [ ]:
# Feature Importance Analysis (for tree-based models)
if 'best_model' in locals() and hasattr(best_model, 'feature_importances_'):
    # Get feature names after preprocessing
    numeric_features_transformed = numeric_features
    categorical_features_transformed = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
    all_features = list(numeric_features_transformed) + list(categorical_features_transformed)
    
    # Get feature importances
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    # Plot top 20 features
    plt.figure(figsize=(12, 8))
    top_n = min(20, len(all_features))
    plt.title(f"Top {top_n} Feature Importances - {best_model_name}")
    plt.barh(range(top_n), importances[indices[:top_n]][::-1])
    plt.yticks(range(top_n), [all_features[i] for i in indices[:top_n]][::-1])
    plt.xlabel("Relative Importance")
    plt.tight_layout()
    plt.show()
    
    # Print top features
    print("Top 20 Most Important Features:")
    for i in range(min(20, len(all_features))):
        print(f"{i+1}. {all_features[indices[i]]}: {importances[indices[i]]:.4f}")

In [ ]:
# Cross-validation for more reliable results
from sklearn.model_selection import cross_val_score

if 'best_model' in locals():
    print(f"\nPerforming 5-fold cross-validation for {best_model_name}...")
    cv_scores = cross_val_score(best_model, X_train_processed, y_train_processed, 
                                cv=5, scoring='accuracy', n_jobs=-1)
    
    print(f"Cross-validation scores: {cv_scores}")
    print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")